# Smartphone Addiction: Analysis and Model Training

This notebook prepares `train.csv` for binary classification of **Addicted Label** (`1` = addicted, `0` = not addicted). It covers:

1. loading and validating the dataset;
2. understanding its structure and quality;
3. basic exploratory analysis;
4. cleaning values and handling missing data; and
5. creating leakage-safe train/test splits and a reusable preprocessing pipeline;
6. comparing candidate classifiers with stratified cross-validation;
7. tuning the strongest candidate; and
8. evaluating the final model once on the untouched internal holdout set;
9. validating a diverse LightGBM blend with out-of-fold predictions; and
10. predicting `test.csv` and creating `sample.csv`.

> Put `train.csv` and `test.csv` beside this notebook or inside the `data/` directory before running all cells.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load the data

Column names are normalized so harmless differences in spaces, punctuation, or capitalization do not break the notebook.

In [ ]:
candidate_paths = [Path("train.csv"), Path("data/train.csv")]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "train.csv was not found. Place it beside this notebook or at data/train.csv."
    )

raw_df = pd.read_csv(data_path)
print(f"Loaded {data_path} with {raw_df.shape[0]:,} rows and {raw_df.shape[1]} columns.")
display(raw_df.head())

In [ ]:
def normalize_column_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")

column_aliases = {
    "id": "participant_id",
    "id_of_participant": "participant_id",
    "age": "age",
    "age_of_participant_in_years": "age",
    "daily_screen_time": "daily_screen_time_hours",
    "daily_screen_time_in_hours": "daily_screen_time_hours",
    "social_media_hours": "social_media_hours",
    "gaming_hours": "gaming_hours",
    "work_study_hours": "work_study_hours",
    "sleep_hours": "sleep_hours",
    "notifications_per_day": "notifications_per_day",
    "app_opens_per_day": "app_opens_per_day",
    "weekend_screen_time": "weekend_screen_time_hours",
    "weekend_screen_time_in_hours": "weekend_screen_time_hours",
    "gender": "gender",
    "gender_as_male_or_female": "gender",
    "stress_level": "stress_level",
    "stress_level_as_low_medium_high": "stress_level",
    "academic_work_impact": "academic_work_impact",
    "academic_work_impact_as_yes_no_null": "academic_work_impact",
    "addicted_label": "addicted_label",
    "addicted_label_as_1_or_0_which_is_the_target": "addicted_label",
}

normalized_columns = [normalize_column_name(column) for column in raw_df.columns]
renamed_columns = [column_aliases.get(column, column) for column in normalized_columns]
if len(set(renamed_columns)) != len(renamed_columns):
    raise ValueError("Column normalization produced duplicate names. Check the CSV headers.")

df = raw_df.copy()
df.columns = renamed_columns

expected_columns = {
    "participant_id", "age", "daily_screen_time_hours",
    "social_media_hours", "gaming_hours", "work_study_hours",
    "sleep_hours", "notifications_per_day", "app_opens_per_day",
    "weekend_screen_time_hours", "gender", "stress_level",
    "academic_work_impact", "addicted_label",
}
missing_columns = expected_columns - set(df.columns)
if missing_columns:
    raise ValueError(
        f"Missing expected columns: {sorted(missing_columns)}. "
        f"Detected columns: {df.columns.tolist()}"
    )

display(pd.DataFrame({"original": raw_df.columns, "normalized": df.columns}))
del raw_df  # Release the duplicate raw frame; this dataset is large.

## 2. Understand structure and data quality

In [ ]:
print("Shape:", df.shape)
display(df.head())
df.info()

In [ ]:
quality_report = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
}).sort_values("missing_percent", ascending=False)

print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate participant IDs: {df['participant_id'].duplicated().sum():,}")
display(quality_report)

In [ ]:
categorical_columns = ["gender", "stress_level", "academic_work_impact"]
for column in categorical_columns + ["addicted_label"]:
    print(f"\n{column}:")
    display(df[column].value_counts(dropna=False).to_frame("count"))

## 3. Clean and validate values

The participant ID identifies a person rather than describing behavior, so it is retained for auditing but excluded from model features. Missing feature values are **not** filled here: imputers will be fitted only on the training split to prevent data leakage.

In [ ]:
numeric_columns = [
    "age", "daily_screen_time_hours", "social_media_hours",
    "gaming_hours", "work_study_hours", "sleep_hours",
    "notifications_per_day", "app_opens_per_day",
    "weekend_screen_time_hours",
]

for column in categorical_columns:
    normalized_values = (
        df[column]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({"": pd.NA, "null": pd.NA, "none": pd.NA, "nan": pd.NA})
    )
    df[column] = normalized_values.astype(object).where(normalized_values.notna(), np.nan)

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["addicted_label"] = pd.to_numeric(df["addicted_label"], errors="coerce")

allowed_categories = {
    "gender": {"male", "female", "other"},
    "stress_level": {"low", "medium", "high"},
    "academic_work_impact": {"yes", "no"},
}
for column, allowed in allowed_categories.items():
    unexpected = set(df[column].dropna().unique()) - allowed
    if unexpected:
        print(f"Warning: unexpected values in {column}: {sorted(unexpected)}")

invalid_target = set(df["addicted_label"].dropna().unique()) - {0, 1}
if invalid_target:
    raise ValueError(f"Target contains values other than 0 and 1: {sorted(invalid_target)}")

missing_target_rows = df["addicted_label"].isna().sum()
if missing_target_rows:
    print(f"Dropping {missing_target_rows:,} rows with a missing/unreadable target.")
    df = df.dropna(subset=["addicted_label"]).copy()
df["addicted_label"] = df["addicted_label"].astype("int8")

print(f"Cleaned shape: {df.shape}")
display(df.head())

In [ ]:
# Plausibility checks flag questionable rows without silently deleting them.
plausible_ranges = {
    "age": (5, 100),
    "daily_screen_time_hours": (0, 24),
    "social_media_hours": (0, 24),
    "gaming_hours": (0, 24),
    "work_study_hours": (0, 24),
    "sleep_hours": (0, 24),
    "notifications_per_day": (0, np.inf),
    "app_opens_per_day": (0, np.inf),
    "weekend_screen_time_hours": (0, 24),
}
range_issues = {}
for column, (minimum, maximum) in plausible_ranges.items():
    issue_count = (~df[column].between(minimum, maximum) & df[column].notna()).sum()
    if issue_count:
        range_issues[column] = int(issue_count)

print("Out-of-range value counts:", range_issues or "None")
display(df[numeric_columns].describe().T)

## 4. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts = df["addicted_label"].value_counts().sort_index()
sns.countplot(data=df, x="addicted_label", ax=axes[0])
axes[0].set(title="Target class counts", xlabel="Addicted label")
axes[1].pie(target_counts, labels=target_counts.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Target class proportions")
plt.tight_layout()
plt.show()

In [ ]:
df[numeric_columns].hist(figsize=(14, 10), bins=25, edgecolor="white")
plt.suptitle("Numeric feature distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for column, axis in zip(categorical_columns, axes):
    sns.countplot(data=df, x=column, hue="addicted_label", ax=axis)
    axis.set_title(f"Addiction label by {column.replace('_', ' ')}")
    axis.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 8))
correlations = df[numeric_columns + ["addicted_label"]].corr()
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric feature correlations")
plt.tight_layout()
plt.show()

## 5. Split the data and build the preprocessing pipeline

The split is stratified to preserve the target ratio. Data-quality analysis found substantial missingness (up to about 19%) and different missing rates in train and external test data, so missingness is retained as model signal instead of being erased. Numeric values are median-imputed with missing indicators; categorical nulls receive their own `missing` category.

The strongest target correlations belong to daily screen time, weekend screen time, and social-media hours. The engineered features below expose meaningful differences, totals, and ratios among these behaviors while preserving the original measurements. All learned preprocessing statistics still come only from the training fold.

In [ ]:
engineered_numeric_columns = [
    "missing_feature_count",
    "average_screen_time_hours",
    "weekend_screen_time_change",
    "recreational_hours",
    "screen_time_component_gap",
    "sleep_screen_balance",
    "daily_engagement_events",
    "notifications_per_app_open",
]

def add_engineered_features(frame):
    base_features = numeric_columns + categorical_columns
    frame["missing_feature_count"] = frame[base_features].isna().sum(axis=1)
    frame["average_screen_time_hours"] = frame[
        ["daily_screen_time_hours", "weekend_screen_time_hours"]
    ].mean(axis=1)
    frame["weekend_screen_time_change"] = (
        frame["weekend_screen_time_hours"] - frame["daily_screen_time_hours"]
    )
    frame["recreational_hours"] = frame[
        ["social_media_hours", "gaming_hours"]
    ].sum(axis=1, min_count=2)
    frame["screen_time_component_gap"] = (
        frame["daily_screen_time_hours"]
        - frame["social_media_hours"]
        - frame["gaming_hours"]
    )
    frame["sleep_screen_balance"] = (
        frame["sleep_hours"] - frame["daily_screen_time_hours"]
    )
    frame["daily_engagement_events"] = frame[
        ["notifications_per_day", "app_opens_per_day"]
    ].sum(axis=1, min_count=2)
    frame["notifications_per_app_open"] = (
        frame["notifications_per_day"]
        / frame["app_opens_per_day"].replace(0, np.nan)
    )
    return frame

df = add_engineered_features(df)
model_numeric_columns = numeric_columns + engineered_numeric_columns

if df["addicted_label"].nunique() != 2:
    raise ValueError("Both target classes (0 and 1) are required for classification.")
if df["addicted_label"].value_counts().min() < 2:
    raise ValueError("Each target class needs at least two rows for a stratified split.")

feature_columns = model_numeric_columns + categorical_columns
X = df[feature_columns].copy()
y = df["addicted_label"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_test)],
    "addicted_rate": [y_train.mean(), y_test.mean()],
}, index=["train", "test"])
display(split_summary)

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, model_numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])

print("Preprocessor is ready. It will be fitted inside each model pipeline.")

## 6. Compare candidate models with stratified cross-validation

Each classifier is wrapped with a fresh copy of `preprocessor`, so imputation, scaling, and encoding are learned separately inside every cross-validation fold.

The dataset is large. Candidate comparison and hyperparameter tuning therefore use a reproducible stratified sample of at most 150,000 training rows. This makes experimentation practical while preserving the class ratio. After selection, the final pipeline is fitted on **all** training rows. The test set remains untouched until final evaluation.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
)

In [ ]:
COMPARISON_SAMPLE_SIZE = 150_000

if len(X_train) > COMPARISON_SAMPLE_SIZE:
    X_model_selection, _, y_model_selection, _ = train_test_split(
        X_train,
        y_train,
        train_size=COMPARISON_SAMPLE_SIZE,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )
else:
    X_model_selection = X_train
    y_model_selection = y_train

print(f"Model-selection rows: {len(X_model_selection):,}")
print(f"Addicted rate: {y_model_selection.mean():.3f}")

In [ ]:
candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=1_000),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "Histogram Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=150,
        random_state=RANDOM_STATE,
    ),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

In [ ]:
comparison_rows = []

for model_name, classifier in candidate_models.items():
    print(f"Evaluating {model_name}...")
    model_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier),
    ])
    scores = cross_validate(
        model_pipeline,
        X_model_selection,
        y_model_selection,
        cv=cv,
        scoring=scoring,
        n_jobs=1,  # Avoid nested parallelism with Random Forest.
        return_train_score=False,
        error_score="raise",
    )
    row = {
        "model": model_name,
        "fit_time_seconds": scores["fit_time"].mean(),
    }
    for metric_name in scoring:
        row[f"{metric_name}_mean"] = scores[f"test_{metric_name}"].mean()
        row[f"{metric_name}_std"] = scores[f"test_{metric_name}"].std()
    comparison_rows.append(row)

comparison_results = (
    pd.DataFrame(comparison_rows)
    .sort_values("roc_auc_mean", ascending=False)
    .reset_index(drop=True)
)
display(comparison_results.round(4))

### How to read the comparison

- **ROC-AUC** measures ranking quality across all classification thresholds and is used below for model selection.
- **Precision** is the proportion of predicted addicted participants that are actually addicted.
- **Recall** is the proportion of addicted participants the model successfully identifies.
- **F1** balances precision and recall.
- The standard deviation indicates how stable each metric is across folds.

If the practical cost of missing an addicted participant is especially high, recall or a custom decision threshold may ultimately matter more than the default ROC-AUC ranking.

## 7. Tune the strongest candidate

The highest mean cross-validated ROC-AUC determines which candidate is tuned. Parameter names begin with `classifier__` because the estimator is inside a pipeline.

In [ ]:
parameter_spaces = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    },
    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 15, 30],
        "classifier__min_samples_leaf": [1, 3, 10],
        "classifier__max_features": ["sqrt", 0.7],
    },
    "Histogram Gradient Boosting": {
        "classifier__learning_rate": [0.03, 0.05, 0.1],
        "classifier__max_iter": [100, 200, 300],
        "classifier__max_leaf_nodes": [15, 31, 63],
        "classifier__min_samples_leaf": [20, 50, 100],
        "classifier__l2_regularization": [0.0, 0.1, 1.0],
    },
}

best_model_name = comparison_results.loc[0, "model"]
selected_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", clone(candidate_models[best_model_name])),
])

parameter_space = parameter_spaces[best_model_name]
total_combinations = int(np.prod([len(values) for values in parameter_space.values()]))
n_iterations = min(10, total_combinations)

print(f"Selected for tuning: {best_model_name}")
print(f"Trying {n_iterations} parameter combinations with 3-fold CV.")

In [ ]:
search = RandomizedSearchCV(
    estimator=selected_pipeline,
    param_distributions=parameter_space,
    n_iter=n_iterations,
    scoring="roc_auc",
    cv=cv,
    refit=True,
    n_jobs=1,
    verbose=1,
    random_state=RANDOM_STATE,
    error_score="raise",
)
search.fit(X_model_selection, y_model_selection)

print(f"Best CV ROC-AUC: {search.best_score_:.4f}")
print("Best parameters:")
display(pd.Series(search.best_params_, name="value").to_frame())

In [ ]:
tuning_results = (
    pd.DataFrame(search.cv_results_)
    .sort_values("rank_test_score")
    [["rank_test_score", "mean_test_score", "std_test_score", "mean_fit_time", "params"]]
    .head(10)
)
display(tuning_results)

## 8. Fit on all training data and evaluate once on the test set

No model-selection decision below uses test performance. The chosen, tuned pipeline is refitted on every training row before this single final evaluation.

In [ ]:
best_pipeline = clone(search.best_estimator_)
best_pipeline.fit(X_train, y_train)

y_test_pred = best_pipeline.predict(X_test)
y_test_probability = best_pipeline.predict_proba(X_test)[:, 1]

test_metrics = pd.Series({
    "accuracy": accuracy_score(y_test, y_test_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "precision": precision_score(y_test, y_test_pred, zero_division=0),
    "recall": recall_score(y_test, y_test_pred, zero_division=0),
    "f1": f1_score(y_test, y_test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_test_probability),
}, name="test_score")

print(f"Final model: {best_model_name}")
display(test_metrics.round(4).to_frame())
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Not addicted (0)", "Addicted (1)"],
    zero_division=0,
))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred, display_labels=["Not addicted", "Addicted"],
    cmap="Blues", ax=axes[0], colorbar=False,
)
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, y_test_probability, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, y_test_probability, ax=axes[2])
axes[2].set_title("Precision-recall curve")
plt.tight_layout()
plt.show()

## Final interpretation checklist

1. Compare final ROC-AUC with 0.5, which represents random ranking.
2. Inspect recall for class `1` to see how many addicted participants are detected.
3. Inspect precision for class `1` to see how reliable positive predictions are.
4. Use the confusion matrix to quantify false negatives and false positives.
5. Choose a different probability threshold later if the costs of false negatives and false positives are unequal.

Do not repeatedly tune against the internal holdout set. If further model development is needed, return to cross-validation on the training data and reserve the holdout set for final confirmation.

## 9. OOF-validated LightGBM ensemble

The public leaderboard is not used to select models or blend weights. Five-fold out-of-fold (OOF) predictions estimate how each model ranks unseen training rows. LightGBM is added because it offers a different tree-building algorithm from histogram gradient boosting and has shown stronger OOF performance on this dataset.

Both probability averaging and rank averaging are tested using only OOF predictions. Rank averaging is especially relevant for ROC-AUC because the metric depends on ordering rather than probability calibration. If blending does not beat the best individual model in OOF AUC, the selected weight naturally falls at an endpoint and uses that model alone. KNN is excluded because its reported OOF AUC is weaker and it scales poorly to this dataset.

In [ ]:
try:
    from lightgbm import LGBMClassifier
except ImportError as exc:
    raise ImportError("Install LightGBM with `%pip install lightgbm` and rerun this cell.") from exc

from scipy.stats import rankdata
from sklearn.model_selection import cross_val_predict

In [ ]:
lightgbm_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LGBMClassifier(
        objective="binary",
        n_estimators=1_000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.90,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )),
])

oof_models = {
    "Tuned baseline": clone(search.best_estimator_),
    "LightGBM": lightgbm_pipeline,
}
ensemble_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
oof_probabilities = {}
oof_rows = []

for model_name, estimator in oof_models.items():
    print(f"Generating 5-fold OOF predictions for {model_name}...")
    probabilities = cross_val_predict(
        estimator,
        X,
        y,
        cv=ensemble_cv,
        method="predict_proba",
        n_jobs=1,  # Each boosting model already uses available CPU cores.
    )[:, 1]
    oof_probabilities[model_name] = probabilities
    oof_rows.append({
        "model": model_name,
        "oof_roc_auc": roc_auc_score(y, probabilities),
    })

oof_results = pd.DataFrame(oof_rows).sort_values("oof_roc_auc", ascending=False)
display(oof_results.round(6))

In [ ]:
baseline_oof = oof_probabilities["Tuned baseline"]
lightgbm_oof = oof_probabilities["LightGBM"]
baseline_oof_rank = rankdata(baseline_oof) / len(baseline_oof)
lightgbm_oof_rank = rankdata(lightgbm_oof) / len(lightgbm_oof)

blend_rows = []
for lightgbm_weight in np.linspace(0, 1, 21):
    baseline_weight = 1 - lightgbm_weight
    probability_blend = (
        baseline_weight * baseline_oof + lightgbm_weight * lightgbm_oof
    )
    rank_blend = (
        baseline_weight * baseline_oof_rank + lightgbm_weight * lightgbm_oof_rank
    )
    blend_rows.extend([
        {
            "blend_space": "probability",
            "lightgbm_weight": lightgbm_weight,
            "baseline_weight": baseline_weight,
            "oof_roc_auc": roc_auc_score(y, probability_blend),
        },
        {
            "blend_space": "rank",
            "lightgbm_weight": lightgbm_weight,
            "baseline_weight": baseline_weight,
            "oof_roc_auc": roc_auc_score(y, rank_blend),
        },
    ])

blend_results = (
    pd.DataFrame(blend_rows)
    .sort_values("oof_roc_auc", ascending=False)
    .reset_index(drop=True)
)
best_blend = blend_results.iloc[0]
display(blend_results.head(10).round(6))
print(
    f"Selected {best_blend['blend_space']} blend: "
    f"LightGBM={best_blend['lightgbm_weight']:.2f}, "
    f"baseline={best_blend['baseline_weight']:.2f}, "
    f"OOF ROC-AUC={best_blend['oof_roc_auc']:.6f}"
)

### Guardrail against leaderboard overfitting

Submit the OOF-selected blend without changing its weights in response to small public-leaderboard movements. The public leaderboard covers only part of the hidden data, so repeatedly adjusting weights against it can reduce private-leaderboard performance. Keep the simpler single model as a second final-submission candidate when Kaggle permits multiple final submissions.

## 10. Predict the external `test.csv` and create `sample.csv`

The `X_test` used above is an internal labeled holdout split. The competition-style `test.csv` loaded here is different: it has participant features but no target. Now that OOF model and blend selection are finished, both models are fitted on **all labeled rows** (`X`, `y`) and combined with the OOF-selected rule.

Kaggle evaluates this competition with **ROC-AUC**, so `addicted_label` must contain predicted probabilities—not thresholded `0/1` classes. Probabilities preserve the model's ranking information and remain in the original test row order.

In [ ]:
test_candidate_paths = [Path("test.csv"), Path("data/test.csv")]
external_test_path = next((path for path in test_candidate_paths if path.exists()), None)
if external_test_path is None:
    raise FileNotFoundError(
        "test.csv was not found. Place it beside this notebook or at data/test.csv."
    )

raw_test_df = pd.read_csv(external_test_path)
test_normalized_columns = [normalize_column_name(column) for column in raw_test_df.columns]
test_renamed_columns = [column_aliases.get(column, column) for column in test_normalized_columns]
if len(set(test_renamed_columns)) != len(test_renamed_columns):
    raise ValueError("Test column normalization produced duplicate names. Check its headers.")

external_test_df = raw_test_df.copy()
external_test_df.columns = test_renamed_columns
required_test_columns = set(numeric_columns + categorical_columns) | {"participant_id"}
missing_test_columns = required_test_columns - set(external_test_df.columns)
if missing_test_columns:
    raise ValueError(
        f"test.csv is missing columns: {sorted(missing_test_columns)}. "
        f"Detected columns: {external_test_df.columns.tolist()}"
    )

print(
    f"Loaded {external_test_path} with {external_test_df.shape[0]:,} rows "
    f"and {external_test_df.shape[1]} columns."
)
display(pd.DataFrame({"original": raw_test_df.columns, "normalized": external_test_df.columns}))
del raw_test_df

In [ ]:
# Apply the same deterministic value cleaning used for train.csv.
for column in categorical_columns:
    normalized_values = (
        external_test_df[column]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({"": pd.NA, "null": pd.NA, "none": pd.NA, "nan": pd.NA})
    )
    external_test_df[column] = (
        normalized_values.astype(object).where(normalized_values.notna(), np.nan)
    )

for column in numeric_columns:
    external_test_df[column] = pd.to_numeric(external_test_df[column], errors="coerce")

external_test_df = add_engineered_features(external_test_df)

test_missing_report = (
    external_test_df[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .to_frame("missing_percent")
)
display(test_missing_report)

In [ ]:
external_test_features = external_test_df[feature_columns]
test_probabilities = {}

for model_name, estimator in oof_models.items():
    print(f"Fitting {model_name} on all {len(X):,} labeled rows...")
    full_estimator = clone(estimator)
    full_estimator.fit(X, y)
    test_probabilities[model_name] = full_estimator.predict_proba(
        external_test_features
    )[:, 1]
    del full_estimator

baseline_test = test_probabilities["Tuned baseline"]
lightgbm_test = test_probabilities["LightGBM"]
baseline_weight = float(best_blend["baseline_weight"])
lightgbm_weight = float(best_blend["lightgbm_weight"])

if best_blend["blend_space"] == "rank":
    baseline_component = rankdata(baseline_test) / len(baseline_test)
    lightgbm_component = rankdata(lightgbm_test) / len(lightgbm_test)
else:
    baseline_component = baseline_test
    lightgbm_component = lightgbm_test

external_test_probabilities = np.clip(
    baseline_weight * baseline_component + lightgbm_weight * lightgbm_component,
    0,
    1,
)

submission = pd.DataFrame({
    "id": external_test_df["participant_id"].to_numpy(),
    "addicted_label": external_test_probabilities,
})

if submission["id"].isna().any():
    raise ValueError("Cannot create submission: test.csv contains missing participant IDs.")
if not submission["addicted_label"].between(0, 1).all():
    raise ValueError("Predicted probabilities must be between 0 and 1.")
if len(submission) != len(external_test_df):
    raise RuntimeError("Submission row count does not match test.csv.")

submission_path = Path("sample.csv")
submission.to_csv(submission_path, index=False)
baseline_submission_path = Path("sample_baseline.csv")
pd.DataFrame({
    "id": external_test_df["participant_id"].to_numpy(),
    "addicted_label": baseline_test,
}).to_csv(baseline_submission_path, index=False)

print(f"Saved {len(submission):,} predictions to {submission_path.resolve()}")
print(f"Saved the single-model fallback to {baseline_submission_path.resolve()}")
display(submission.head())
display(submission["addicted_label"].describe().to_frame("predicted_probability"))